# PyCAM-SIMA persistent pool: reuse and fork

This Notebook keeps one MPI pool alive across cells. The persistent path launches MPI once, initializes the base once, and forks three children into pre-allocated slots. Jupyter displays each cell's execution duration, so the code contains no manual timers.

## 1. Execution paths

```text
Persistent pool                            Legacy multi-job
one launcher worker + one large mpiexec     base MPI job
├── worker A: base Actor       → slot 0     ├── child MPI job: control
├── worker B: control Actor    → slot 1     ├── child MPI job: no-kessler
├── worker C: no-kessler Actor → slot 2     └── child MPI job: warm
└── worker D: warm Actor       → slot 3

fork = MPI rank-to-rank memory copy     fork = checkpoint + new MPI startup
```

Run the persistent cells and inspect Jupyter's cell duration. The optional legacy cell performs the corresponding multi-job workload, and its cell duration includes real queue, MPI startup, checkpoint, and model costs.

## 2. Configure Dask and calculate the pool

In [ ]:
from datetime import datetime
from pathlib import Path
import os
import shutil

import numpy as np
import pycam_sima
from dask.distributed import Client
from pycam_sima import DaskExperimentClient

repo = Path('/glade/work/ruitong/pycam-sima')
scratch = Path(os.environ.get('SCRATCH', '/glade/derecho/scratch/ruitong'))
config_path = repo / 'configs/fkessler_model.yaml'
reference_atm_in = repo / 'reference/cases/FKESSLER_ne3pg3_gnu_24x50/CaseDocs/atm_in'
stamp = datetime.now().strftime('%Y%m%d-%H%M%S')
experiment_root = scratch / 'pycam-sima/persistent_pool_trials' / stamp
initial_run_dir = experiment_root / 'initial-run'
initial_run_dir.mkdir(parents=True, exist_ok=False)
shutil.copy2(reference_atm_in, initial_run_dir / 'atm_in')

# Use allocation mode when this Jupyter server is running inside qsub -I.
execution_mode = 'allocation' if os.environ.get('PBS_JOBID') else 'pbs'
client = Client(
    processes=False,
    # One launcher worker plus one ModelActor worker per possible slot.
    n_workers=5,
    threads_per_worker=1,
    dashboard_address=None,
)
experiments = DaskExperimentClient(
    client,
    config=config_path,
    initial_run_dir=initial_run_dir,
    run_root=experiment_root / 'models',
    python_executable=repo / '.venv/bin/python',
    execution_mode=execution_mode,
)
resource_plan = experiments.plan_pool(
    max_concurrent_models=4,
    ranks_per_model=None,  # inherit mpi_size=24 from ModelConfig
    memory_per_model='auto',
)
{
    'pycam_sima': pycam_sima.__version__,
    'execution_mode': execution_mode,
    'run_root': str(experiment_root),
    'resource_plan': resource_plan.describe(),
}

{'pycam_sima': '0.18.0',
 'execution_mode': 'pbs',
 'run_root': '/glade/derecho/scratch/ruitong/pycam-sima/persistent_pool_trials/20260729-225308',
 'resource_plan': {'available_nodes': 4,
  'available_cpus': 96,
  'available_memory_bytes': 343597383680,
  'ranks_per_model': 24,
  'model_slots': 4,
  'world_size': 96,
  'slot_placements': ((0,
    1,
    2,
    3,
    4,
    5,
    6,
    7,
    8,
    9,
    10,
    11,
    12,
    13,
    14,
    15,
    16,
    17,
    18,
    19,
    20,
    21,
    22,
    23),
   (24,
    25,
    26,
    27,
    28,
    29,
    30,
    31,
    32,
    33,
    34,
    35,
    36,
    37,
    38,
    39,
    40,
    41,
    42,
    43,
    44,
    45,
    46,
    47),
   (48,
    49,
    50,
    51,
    52,
    53,
    54,
    55,
    56,
    57,
    58,
    59,
    60,
    61,
    62,
    63,
    64,
    65,
    66,
    67,
    68,
    69,
    70,
    71),
   (72,
    73,
    74,
    75,
    76,
    77,
    78,
    79,
    80,
    81,
    82,
    

PyCAM-SIMA pool submitted as 6955626.desched1; waiting for 4 x 24 MPI ranks ...


## 3. Start the pool and base once

The pool and base are stored as normal Python variables, so later cells reuse the same live MPI processes and StatePool. Do not rerun this cell without first running the cleanup cell.

In [2]:
pool = experiments.pool('cam-pool', resource_plan=resource_plan)

base = pool.model('base')

pool_started = pool.status
base_started = base.status
assert pool_started['mpi_launch_count'] == 1
{
    'mpi_launch_count': pool_started['mpi_launch_count'],
    'pool_mpi_launch_id': pool_started.get('pool_mpi_launch_id'),
    'pbs_job_id': pool_started.get('pbs_job_id'),
    'launcher_worker': pool.worker,
    'base_worker': base.worker,
    'base_slot': base.slot_id,
    'scheduler': pool.scheduler_status,
    'base_status': base_started,
    'slots': pool.slots,
}

{'mpi_launch_count': 1,
 'pool_mpi_launch_id': 'cam-pool-derecho1-104338-1785387189227970894',
 'pbs_job_id': None,
 'launcher_worker': 'inproc://128.117.211.170/104338/10',
 'base_worker': 'inproc://128.117.211.170/104338/12',
 'base_slot': 0,
 'scheduler': {'launcher_worker': 'inproc://128.117.211.170/104338/10',
  'model_workers': {'base': 'inproc://128.117.211.170/104338/12'},
  'available_workers': ('inproc://128.117.211.170/104338/10',
   'inproc://128.117.211.170/104338/12',
   'inproc://128.117.211.170/104338/4',
   'inproc://128.117.211.170/104338/6',
   'inproc://128.117.211.170/104338/8'),
  'actor_layout': 'model-per-worker',
  'worker_policy': 'exclusive'},
 'base_status': ModelStatus(name='base', running=True, ranks=24, step=0, native_calls=0, mpi_launch_count=1, worker_host='derecho1', worker_pid=104338, launch_mode='pbs', pbs_job_id='6955626.desched1', outer_pbs_job_id=None, field_count=360, snapshot_transport='initialization', run_dir=PosixPath('/glade/derecho/scratch/

## 4. A later cell reuses the same base memory

No new `mpiexec`, no model restart, and no checkpoint restore occurs here. This cell demonstrates dynamic field creation/removal and runtime Fortran plugin loading before the fork.

In [ ]:
launches_before_reuse = pool.status['mpi_launch_count']
base.advance(steps=2)

base.fields.create(
    'experiment_tracer',
    dims=('column', 'level'),
    units='kg kg-1',
    initial=0.0,
)
base.fields.create('temporary_probe', dims=('column',), initial=1.0)
deleted_probe = base.fields.delete('temporary_probe')
installed_plugin = base.physics.install(
    source=repo / 'examples/plugins/runtime_temperature_offset/device.yaml',
    project_root=repo,
    after='kessler',
    inputs={
        'runtime_plugin_temperature': 240.0,
        'runtime_plugin_temperature_increment': 1.5,
    },
)
plugin_field = base.fields.ccpp_runtime_plugin_temperature
plugin_before = plugin_field.stats(rank=0)
base.physics.scheme('runtime_temperature_offset', group='before').run()
plugin_after = plugin_field.stats(rank=0)
assert np.isclose(plugin_after['mean'] - plugin_before['mean'], 1.5)
assert pool.status['mpi_launch_count'] == launches_before_reuse == 1
{
    'base_step': base.status.step,
    'same_mpi_launch_count': pool.status['mpi_launch_count'],
    'deleted_dynamic_field': deleted_probe['standard_name'],
    'installed_plugin': installed_plugin['name'],
    'plugin_increment': plugin_after['mean'] - plugin_before['mean'],
}

{'base_step': 2,
 'same_mpi_launch_count': 1,
 'deleted_dynamic_field': 'temporary_probe',
 'installed_plugin': 'runtime_temperature_offset',
 'plugin_increment': 1.5}

## 5. Insert a Notebook Python function into the live suite

`install_python()` serializes this trusted function and installs it on every rank of the base slot. `reads` are read-only views, `writes` are writable rank-local views, and only declared fields are visible. The returned handle can run, disable, enable, move, or remove the process. The snapshot records the exact function payload and hash.

In [ ]:
def notebook_tracer_source(fields, context):
    tracer = fields['experiment_tracer']
    tracer[...] += context.timestep_seconds * 1.0e-6

python_process = base.physics.install_python(
    notebook_tracer_source,
    name='notebook_tracer_source',
    group='physics_before_coupler',
    after='kessler',
    writes=('experiment_tracer',),
)

# Run only this process: the model clock does not advance.
tracer_before = base.fields.experiment_tracer.get(rank=0)
step_before = base.status.step
python_process.run()
tracer_after = base.fields.experiment_tracer.get(rank=0)
assert base.status.step == step_before
assert np.allclose(
    tracer_after,
    tracer_before + 1800.0e-6,
    rtol=0.0,
    atol=np.spacing(1800.0e-6),
)

# A disabled process is skipped by a complete step; enabling restores it.
python_process.disable()
disabled_before = base.fields.experiment_tracer.get(rank=0)
base.advance(steps=1)
disabled_after = base.fields.experiment_tracer.get(rank=0)
assert np.array_equal(disabled_before, disabled_after)
python_process.enable()
base.advance(steps=1)
enabled_after = base.fields.experiment_tracer.get(rank=0)
assert np.all(enabled_after > disabled_after)

# The disk checkpoint metadata retains python_process_inventory.
python_process_checkpoint = base.save()
installed_python_processes = base.status.details['python_processes']
{
    'process': python_process.name,
    'payload_hash': python_process.payload_hash,
    'writes': python_process.writes,
    'step': base.status.step,
    'checkpoint': python_process_checkpoint.path,
    'inventory': installed_python_processes,
}

## 6. Fork three models into pre-allocated slots

The child MPI ranks already exist. Fork copies each parent rank's StatePool directly to the matching rank in an idle slot. Jupyter's duration for this cell is the fork cost.

In [4]:
branches = base.fork(
    'control', 'no_kessler', 'warm', require_concurrent=True
)

branches.no_kessler.physics.kessler.disable()
branches.warm.fields.air_temperature += 1.0
control_temperature = branches.control.fields.air_temperature.get(rank=0)
warm_temperature = branches.warm.fields.air_temperature.get(rank=0)
assert np.array_equal(warm_temperature, np.add(control_temperature, 1.0))
assert pool.status['mpi_launch_count'] == 1
{
    'mpi_launch_count_after_fork': pool.status['mpi_launch_count'],
    'branch_statuses': branches.statuses,
    'occupied_slots': pool.slots,
}

{'mpi_launch_count_after_fork': 1,
 'branch_statuses': {'control': ModelStatus(name='control', running=True, ranks=24, step=2, native_calls=710, mpi_launch_count=1, worker_host='derecho1', worker_pid=104338, launch_mode='pbs', pbs_job_id='6955626.desched1', outer_pbs_job_id=None, field_count=363, snapshot_transport='mpi', run_dir=PosixPath('/glade/derecho/scratch/ruitong/pycam-sima/persistent_pool_trials/20260729-225308/models/cam-pool/control/run'), history_dir=PosixPath('/glade/derecho/scratch/ruitong/pycam-sima/persistent_pool_trials/20260729-225308/models/cam-pool/control/history'), log_path=PosixPath('/glade/work/ruitong/pycam-sima/logs/pycam_pool_cam-pool.log'), details={'step': 2, 'native_nstep': 2, 'native_calls': 710, 'phase_status': {'runtime': 'model', 'state': 'RUNNING', 'last_phase': 'physics_timestep_initial', 'last_scheme': 'plugin:runtime_temperature_offset.runtime_temperature_offset@25', 'last_scheme_group': 'physics_before_coupler', 'next_phase': None, 'sequence_safe'

In [ ]:
# The callback payload, placement, permissions, and enabled state were
# copied inside the MPI fork payload. Each child owns an independent registry.
control_inventory = branches.control.status.details['python_processes']
assert any(
    item['spec']['name'] == 'notebook_tracer_source'
    for item in control_inventory
)
parent_tracer = base.fields.experiment_tracer.get(rank=0)
control_tracer_before = branches.control.fields.experiment_tracer.get(rank=0)
branches.control.physics.scheme(
    'notebook_tracer_source', group='physics_before_coupler'
).run()
control_tracer_after = branches.control.fields.experiment_tracer.get(rank=0)
assert np.all(control_tracer_after > control_tracer_before)
assert np.array_equal(base.fields.experiment_tracer.get(rank=0), parent_tracer)

removed_from_warm = branches.warm.physics.remove_python(
    'notebook_tracer_source'
)
assert branches.control.physics.scheme(
    'notebook_tracer_source', group='physics_before_coupler'
).enabled
{
    'inherited_payload_hash': control_inventory[0]['spec']['payload_hash'],
    'control_changed_independently': True,
    'parent_unchanged': True,
    'removed_only_from_warm': removed_from_warm['name'],
}

## 7. Run one scheme on one selected branch

This calls only Kessler on the `control` model's slot. It does not execute the surrounding suite order, advance the model clock, write a complete-step history record, or change the other branches. Use this fine-grained call for diagnostics and controlled experiments.

In [ ]:
control_step_before_kessler = branches.control.status.step

kessler_result = branches.control.physics.scheme(
    'kessler',
    group='physics_before_coupler',
).run()

control_status_after_kessler = branches.control.status
assert control_status_after_kessler.step == control_step_before_kessler
{
    'target_model': branches.control.name,
    'target_slot': branches.control.slot_id,
    'step_unchanged': control_status_after_kessler.step,
    'kessler_result': kessler_result,
}

## 8. Another cell continues the same three children

The handles and all StatePools remain live from the previous cell. Each submit call creates a Dask Future on that model's own worker. The Launcher batches ready commands for different slots into one MPI-world command.

In [5]:
steps_before = {name: status.step for name, status in branches.statuses.items()}
control_future = branches.control.submit.advance(steps=1)
no_kessler_future = branches.no_kessler.submit.advance(steps=1)
warm_future = branches.warm.submit.advance(steps=1)
client.gather((control_future, no_kessler_future, warm_future))

# This observation is an explicit child of warm_future in Dask's graph.
warm_stats_future = branches.warm.submit.fields.air_temperature.stats(
    rank=0,
    depends_on=warm_future,
)
warm_stats = warm_stats_future.result()
steps_after = {name: status.step for name, status in branches.statuses.items()}
assert all(steps_after[name] == steps_before[name] + 1 for name in steps_before)
assert pool.status['mpi_launch_count'] == 1
{
    'steps_before': steps_before,
    'steps_after': steps_after,
    'model_workers': {
        name: model.worker for name, model in branches.items()
    },
    'model_slots': {
        name: model.slot_id for name, model in branches.items()
    },
    'warm_temperature': warm_stats,
    'mpi_launch_count': pool.status['mpi_launch_count'],
}

{'steps_before': {'control': 2, 'no_kessler': 2, 'warm': 2},
 'steps_after': {'control': 3, 'no_kessler': 3, 'warm': 3},
 'model_workers': {'control': 'inproc://128.117.211.170/104338/4',
  'no_kessler': 'inproc://128.117.211.170/104338/6',
  'warm': 'inproc://128.117.211.170/104338/8'},
 'model_slots': {'control': 1, 'no_kessler': 2, 'warm': 3},
 'warm_temperature': {'rank': 0,
  'shape': (4, 4, 30, 3, 3),
  'dtype': '<f8',
  'min': 150.84296723573559,
  'max': 307.22263051472413,
  'mean': 238.25173907805524},
 'mpi_launch_count': 1}

## 9. Optional legacy multi-job comparison

Set the flag to `True` only when the Notebook runs in `execution_mode='pbs'`. This intentionally submits one legacy base job plus three child jobs. Compare this cell's Jupyter duration with the persistent pool cells above. It is disabled by default to prevent accidental extra PBS submissions.

In [6]:
run_legacy_comparison = False

if not run_legacy_comparison:
    legacy_result = 'skipped; set run_legacy_comparison=True for the real four-job comparison'
elif execution_mode != 'pbs':
    raise RuntimeError('legacy multi-model timing requires execution_mode="pbs"')
else:
    legacy_client = Client(
        processes=False,
        n_workers=4,
        threads_per_worker=1,
        dashboard_address=None,
    )
    legacy_experiments = DaskExperimentClient(
        legacy_client,
        config=config_path,
        initial_run_dir=initial_run_dir,
        run_root=experiment_root / 'legacy-benchmark',
        python_executable=repo / '.venv/bin/python',
        execution_mode='pbs',
    )
    try:
        with legacy_experiments.model('legacy-base') as legacy_base:
            legacy_base.advance(steps=2)
            control_plan = legacy_experiments.plan('legacy-control')
            no_kessler_plan = legacy_experiments.plan('legacy-no-kessler', experimental=True)
            no_kessler_plan.physics.kessler.disable()
            warm_plan = legacy_experiments.plan('legacy-warm')
            warm_plan.fields.edit('air_temperature', 'add', 1.0)

            legacy_children = legacy_experiments.fork_models(
                legacy_base,
                (control_plan, no_kessler_plan, warm_plan),
                close_parent=False,
            )
            with legacy_children:
                legacy_children.advance(steps=1)
                legacy_statuses = legacy_children.statuses
                legacy_child_job_ids = {
                    name: status.pbs_job_id
                    for name, status in legacy_statuses.items()
                }
            legacy_base_job_id = legacy_base.status.pbs_job_id

        legacy_result = {
            'base_pbs_job_id': legacy_base_job_id,
            'child_pbs_job_ids': legacy_child_job_ids,
        }
    finally:
        legacy_client.close()

legacy_result

'skipped; set run_legacy_comparison=True for the real four-job comparison'

## 10. Restore the Python process from the disk checkpoint

Close the child slots, restore the earlier base checkpoint into one of those already-running slots, and call the restored function. This reads shared checkpoint files but does not submit PBS or launch MPI again.

In [ ]:
branches.close()
with pool.restore(
    'checkpoint-restart', python_process_checkpoint.path
) as restarted:
    restored_inventory = restarted.status.details['python_processes']
    assert any(
        item['spec']['name'] == 'notebook_tracer_source'
        for item in restored_inventory
    )
    restored_before = restarted.fields.experiment_tracer.get(rank=0)
    restarted.physics.scheme(
        'notebook_tracer_source', group='physics_before_coupler'
    ).run()
    restored_after = restarted.fields.experiment_tracer.get(rank=0)
    assert np.all(restored_after > restored_before)
    restored_step = restarted.status.step

assert pool.status['mpi_launch_count'] == 1
{
    'restored_step': restored_step,
    'callback_restored': True,
    'mpi_launch_count': pool.status['mpi_launch_count'],
}

## 11. Cleanup

Run this cell when finished. It closes the three children, base, pool MPI world, and Dask client in reverse creation order.

In [7]:
branches.close()
base.close()
pool.close()
client.close()
{
    'children_closed': True,
    'base_closed': True,
    'pool_closed': True,
    'dask_client_closed': True,
}

{'children_closed': True,
 'base_closed': True,
 'pool_closed': True,
 'dask_client_closed': True}